# StylePops — Kombin + A/B Üretimi (Colab GPU)

Mac'te kombin üretme. Colab GPU ile ~3–5 dk.

## Mac (bir kez)
```bash
python scripts/pack_colab_combo_bundle.py
```
→ `outputs/colab_combo_bundle.zip` (~29 MB)

## Colab
1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Factory reset runtime** (eski oturumu temizle)
3. **Runtime → Run all**
4. **"Dosya seç"** ile Mac'teki `colab_combo_bundle.zip` yükle — Drive gerekmez

In [ ]:
import torch
assert torch.cuda.is_available(), 'Runtime → GPU (T4) seç!'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q torch torchvision transformers accelerate peft pillow joblib lightgbm

In [ ]:
import os, shutil
from pathlib import Path

PROJECT = Path('/content/StylePops_Modules')
os.chdir('/content')
if PROJECT.exists():
    shutil.rmtree(PROJECT)
!git clone -q https://github.com/valueisinvalid/StylePops_Modules.git
os.chdir(PROJECT)
!git pull -q
print('Repo güncel:', PROJECT)

In [ ]:
# Mac'teki outputs/colab_combo_bundle.zip dosyasını seç (Drive gerekmez)
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if not uploaded:
    raise FileNotFoundError('Zip seçilmedi. Mac: outputs/colab_combo_bundle.zip')
ZIP_PATH = Path('/content') / next(iter(uploaded))
print('Yüklendi:', ZIP_PATH, f'({ZIP_PATH.stat().st_size // 1024 // 1024} MB)')

In [ ]:
from pathlib import Path
import zipfile, shutil, json

PROJECT = Path('/content/StylePops_Modules')
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall('/content')

SRC = Path('/content/stylepops_colab')
for rel in ('data/visual/garments_production.json', 'data/visual/inventory_registry.json'):
    shutil.copy2(SRC / rel, PROJECT / rel)
shutil.copytree(SRC / 'data' / 'lookups', PROJECT / 'data' / 'lookups', dirs_exist_ok=True)
for folder in ('data/assets/garments', 'data/assets/fashion_product'):
    src_dir = SRC / folder
    if src_dir.exists():
        shutil.copytree(src_dir, PROJECT / folder, dirs_exist_ok=True)
lora_src = SRC / 'outputs' / 'fashionclip_lora'
if lora_src.exists():
    shutil.copytree(lora_src, PROJECT / 'outputs' / 'fashionclip_lora', dirs_exist_ok=True)
    print('LoRA ✓')

n = json.loads((PROJECT / 'data/visual/garments_production.json').read_text())['count']
print(f'Gardırop hazır: {n} parça')

In [ ]:
import os, runpy, sys
from pathlib import Path

os.chdir('/content/StylePops_Modules')
Path('data/assets/combos').mkdir(parents=True, exist_ok=True)
os.environ['PYTHONUNBUFFERED'] = '1'
sys.argv = ['generate_visual_combinations.py', '--ab-pairs', '200']
runpy.run_path('scripts/generate_visual_combinations.py', run_name='__main__')

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

root = Path('/content/StylePops_Modules')
out = Path('/content/stylepops_results.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in ('data/visual/combinations_visual.csv', 'data/visual/ab_pairs.csv'):
        zf.write(root / name, Path(name).name)

combos_zip = Path('/content/stylepops_combos.zip')
!cd /content/StylePops_Modules/data/assets/combos && zip -qr {combos_zip} AB*.jpg VC*.jpg

print('İndiriliyor… Mac\'te:')
print('  combinations_visual.csv + ab_pairs.csv → data/visual/')
print('  AB*.jpg VC*.jpg → data/assets/combos/')
files.download(str(out))
files.download(str(combos_zip))